In [5]:
import pandas as pd
df = pd.read_csv("../data/raw/genetic_disorder_raw.csv")
df.shape
df.info()
df.isna().mean().sort_values(ascending=False) * 100
df["Genetic Disorder"].value_counts()
df["Disorder Subclass"].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18000 entries, 0 to 17999
Data columns (total 33 columns):
 #   Column                                            Non-Null Count  Dtype  
---  ------                                            --------------  -----  
 0   Patient Id                                        18000 non-null  object 
 1   Patient Age                                       16913 non-null  float64
 2   Genes in mother's side                            18000 non-null  object 
 3   Inherited from father                             17774 non-null  object 
 4   Maternal gene                                     15808 non-null  object 
 5   Paternal gene                                     18000 non-null  object 
 6   Blood cell count (mcL)                            18000 non-null  float64
 7   Patient First Name                                18000 non-null  object 
 8   Family Name                                       10494 non-null  object 
 9   Father's name    

Disorder Subclass
Diabetes                               4780
Leigh syndrome                         2740
Cystic fibrosis                        2482
Mitochondrial myopathy                 2340
Tay-Sachs                              1974
Hemochromatosis                         966
Alzheimer's                             411
Leber's hereditary optic neuropathy     347
Cancer                                  264
Name: count, dtype: int64

In [6]:
# 1. Overall missing-value % per column (full list, not just what showed above)
missing_pct = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
print(missing_pct)

Family Name                                         41.7
Father's age                                        25.7
Mother's age                                        25.4
Autopsy shows birth defect (if applicable)          19.7
Maternal gene                                       12.2
H/O substance abuse                                  9.6
Follow-up                                            9.4
Symptom 5                                            9.4
Disorder Subclass                                    9.4
Assisted conception IVF/ART                          9.2
Heart Rate (rates/min                                9.2
Birth defects                                        9.2
Folic acid details (peri-conceptional)               9.2
Symptom 1                                            9.1
Blood test result                                    9.1
White Blood cell count (thousand per microliter)     9.1
H/O radiation exposure (x-ray)                       9.1
Symptom 2                      

In [7]:
# 2. Genetic Disorder (main target) balance
df["Genetic Disorder"].value_counts()

Genetic Disorder
Mitochondrial genetic inheritance disorders     6000
Multifactorial genetic inheritance disorders    6000
Single-gene inheritance diseases                6000
Name: count, dtype: int64

In [8]:
# 3. Drop identity-style columns early so you don't accidentally EDA on them 
identity_cols = ["Patient Id", "Patient First Name", "Family Name", "Father's name"]
df_clean = df.drop(columns=identity_cols)
df_clean.shape

(18000, 29)

In [9]:
# 4. Check numeric column distributions 
df_clean.describe()

,Patient Age,Blood cell count (mcL),Mother's age,Father's age,White Blood cell count (thousand per microliter),Symptom 1,Symptom 2,Symptom 3,Symptom 4,Symptom 5
count,16913.000000,18000.000000,13431.000000,13371.000000,16367.000000,16355.000000,16380.000000,16373.000000,16392.000000,16317.000000
mean,6.978596,4.897941,34.640756,42.058859,7.481969,0.628799,0.607326,0.605387,0.570278,0.550837
std,4.328421,0.199140,9.831324,13.052591,2.674020,0.483141,0.488360,0.488782,0.495051,0.497424
min,0.000000,4.185821,18.000000,20.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,3.000000,4.764406,26.000000,31.000000,5.389703,0.000000,0.000000,0.000000,0.000000,0.000000
50%,7.000000,4.897487,35.000000,42.000000,7.441468,1.000000,1.000000,1.000000,1.000000,1.000000
75%,11.000000,5.033884,43.000000,53.000000,9.558073,1.000000,1.000000,1.000000,1.000000,1.000000
max,14.000000,5.609829,51.000000,64.000000,12.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [10]:
# 5. Check categorical column unique values (helps spot messy/inconsistent labels)
for col in df_clean.select_dtypes(include="object").columns:
    print(col, "->", df_clean[col].nunique(), "unique values")

Genes in mother's side -> 2 unique values
Inherited from father -> 2 unique values
Maternal gene -> 2 unique values
Paternal gene -> 2 unique values
Status -> 2 unique values
Respiratory Rate (breaths/min) -> 2 unique values
Heart Rate (rates/min -> 2 unique values
Follow-up -> 2 unique values
Gender -> 3 unique values
Autopsy shows birth defect (if applicable) -> 3 unique values
Folic acid details (peri-conceptional) -> 2 unique values
H/O serious maternal illness -> 2 unique values
H/O radiation exposure (x-ray) -> 4 unique values
H/O substance abuse -> 4 unique values
Assisted conception IVF/ART -> 2 unique values
Birth defects -> 2 unique values
Blood test result -> 4 unique values
Genetic Disorder -> 3 unique values
Disorder Subclass -> 9 unique values


In [11]:
# Separate your columns into numeric vs categorical
numeric_cols = df_clean.select_dtypes(include=["float64", "int64"]).columns
categorical_cols = df_clean.select_dtypes(include="object").columns

print("Numeric:", list(numeric_cols))
print("Categorical:", list(categorical_cols))

Numeric: ['Patient Age', 'Blood cell count (mcL)', "Mother's age", "Father's age", 'White Blood cell count (thousand per microliter)', 'Symptom 1', 'Symptom 2', 'Symptom 3', 'Symptom 4', 'Symptom 5']
Categorical: ["Genes in mother's side", 'Inherited from father', 'Maternal gene', 'Paternal gene', 'Status', 'Respiratory Rate (breaths/min)', 'Heart Rate (rates/min', 'Follow-up', 'Gender', 'Autopsy shows birth defect (if applicable)', 'Folic acid details (peri-conceptional)', 'H/O serious maternal illness', 'H/O radiation exposure (x-ray)', 'H/O substance abuse', 'Assisted conception IVF/ART', 'Birth defects', 'Blood test result', 'Genetic Disorder', 'Disorder Subclass']


In [13]:
# Fill missing numeric values with the median
# (Median is safer than mean here - it's not thrown off by outliers)

for col in numeric_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

In [14]:
# Fill missing categorical values with the mode (most common value)

for col in categorical_cols:
    if df_clean[col].isna().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

In [15]:
# Confirm no missing values remain 

df_clean.isna().sum().sum() # should be 0

np.int64(0)

In [16]:
# Save the cleaned dataset 

df_clean.to_csv("../data/processed/genetic_disorder_cleaned.csv", index=False)

In [17]:
# Check for duplicate rows 
df_clean.duplicated().sum()

np.int64(4015)

In [18]:
before = df_clean["Genetic Disorder"].value_counts()
after = df_clean.drop_duplicates()["Genetic Disorder"].value_counts()
print("Before:\n", before)
print("\nAfter:\n", after)

Before:
 Genetic Disorder
Mitochondrial genetic inheritance disorders     6000
Multifactorial genetic inheritance disorders    6000
Single-gene inheritance diseases                6000
Name: count, dtype: int64

After:
 Genetic Disorder
Mitochondrial genetic inheritance disorders     6000
Single-gene inheritance diseases                6000
Multifactorial genetic inheritance disorders    1985
Name: count, dtype: int64


In [19]:
# encoding categorical columns 
from sklearn.preprocessing import LabelEncoder 

df_encoded = df_clean.copy()
encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
    encoders[col] = le

df_encoded.head()

,Patient Age,Genes in mother's side,Inherited from father,Maternal gene,Paternal gene,Blood cell count (mcL),Mother's age,Father's age,Status,Respiratory Rate (breaths/min),...,Birth defects,White Blood cell count (thousand per microliter),Blood test result,Symptom 1,Symptom 2,Symptom 3,Symptom 4,Symptom 5,Genetic Disorder,Disorder Subclass
0,6.0,1,0,0,0,5.025882,31.0,59.0,1,1,...,1,8.723627,0,1.0,1.0,0.0,0.0,0.0,0,6
1,7.0,1,1,0,1,5.006373,41.0,46.0,1,1,...,1,6.597532,0,0.0,1.0,1.0,1.0,1.0,1,3
2,9.0,1,1,1,0,4.928363,40.0,42.0,1,0,...,1,9.578838,2,1.0,0.0,0.0,1.0,1.0,1,3
3,8.0,1,0,1,1,4.768017,27.0,42.0,0,1,...,0,12.000000,3,1.0,0.0,1.0,0.0,1.0,0,7
4,0.0,0,1,1,0,4.844957,21.0,49.0,0,0,...,0,7.441468,1,1.0,0.0,0.0,1.0,1.0,0,6


In [20]:
df_encoded.to_csv("../data/processed/genetic_disorder_encoded.csv", index=False)